# Car Parts Segmentation Training

This notebook trains a YOLO26 instance-segmentation model for vehicle part localization. The resulting checkpoint is used by the API to attach a car part, such as `front_bumper` or `hood`, to each detected damage region.


## Setup

Install the runtime dependencies, verify the environment, and confirm that a GPU is available when running in Colab or another hosted notebook environment.


In [ ]:
%pip install -q ultralytics==8.4.78 opencv-python-headless pyyaml

import ultralytics
ultralytics.checks()


## Dataset

The training data is the car-parts segmentation dataset used by the Ultralytics dataset configuration. It contains masks for 23 part classes and can be downloaded automatically by passing `carparts-seg.yaml` to training.

Key classes include bumpers, doors, glass, lights, hood, mirrors, tailgate, trunk, and wheels.


In [ ]:
DATA_YAML = "carparts-seg.yaml"
MODEL_NAME = "yolo26n-seg.pt"
EPOCHS = 100
IMG_SIZE = 640
BATCH = 16
PROJECT_DIR = "runs/car_parts_segmentation"
RUN_NAME = "car_parts_yolo26"


## Train

Start with `yolo26n-seg.pt` for a fast baseline. For a stronger production checkpoint, change `MODEL_NAME` to `yolo26s-seg.pt`, `yolo26m-seg.pt`, or a larger YOLO26 segmentation variant and adjust batch size to fit the GPU.


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project=PROJECT_DIR,
    name=RUN_NAME,
    seed=42,
    deterministic=True,
)


## Validate

Validate the best checkpoint and inspect both box and mask metrics before using it in the API.


In [ ]:
best_model_path = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML, imgsz=IMG_SIZE)

print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Box mAP50-95: {metrics.box.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")


## Predict

Run a quick visual check on a sample image. Replace `SAMPLE_IMAGE` with an internal validation image when evaluating the model for deployment.


In [ ]:
SAMPLE_IMAGE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/carparts-image.jpg"
prediction_results = best_model.predict(SAMPLE_IMAGE, imgsz=IMG_SIZE, save=True)


## Export

The API can load the PyTorch `best.pt` checkpoint directly. Export only when targeting a separate runtime such as ONNX, OpenVINO, or TensorRT.


In [ ]:
# Optional export for non-PyTorch deployment targets.
# exported_path = best_model.export(format="onnx", imgsz=IMG_SIZE)
# print(exported_path)
